# 02 · Natural-class pseudo-labels for BePLi v2 with Mask2Former

BePLi v2 only annotates plastic. To use it as 3-class training data, the natural class is predicted by a Mask2Former model trained on BeachLitter v2022.

1. Train a 3-class Mask2Former (0 background, 1 artificial, 2 natural) on v2022 only.
2. Tune the learning rate briefly, then train the final model with the best one.
3. Evaluate once on the v2022 test set.
4. Predict the natural class on the 2,152 BePLi images and merge it with BePLi's own plastic masks (plastic always wins) into 0/1/2 masks.

**Training settings**
- Mask2Former (Cheng et al., 2022): learning rate 1e-4, weight decay 0.05, backbone learning-rate multiplier 0.1, 512 × 512 crop for Swin-S, 100 queries.
- MaskFormer (Cheng et al., 2021), the ADE20K recipe Mask2Former builds on: AdamW, poly schedule, random scale 0.5–2.0, horizontal flip, random crop, colour jitter; learning rate 6e-5 for Swin backbones.
- Only the learning rate is tuned, over three values from these references: 1e-4 (Mask2Former), 6e-5 (MaskFormer, Swin) and 5e-5 (1e-4 scaled linearly from batch 16 to the effective batch of 8; Goyal et al., 2017). Everything else follows the papers.

Run with **Save Version → Save & Run All** (GPU, Internet on).

## 1. Packages and imports

In [ ]:
!pip install -q pycocotools

import gc
import json
import math
import random
import re
import shutil
import time
from pathlib import Path

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pycocotools import mask as mask_utils
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import Mask2FormerForUniversalSegmentation

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## 2. Configuration

`CKPT_INPUT = None` runs tuning and training from scratch. Set it to a saved `mask2former_v2022_best.pt` to skip both and go straight to evaluation and pseudo-labelling.

In [ ]:
V2022_IMG = Path("/kaggle/input/datasets/rafiadabhi/beach-litter/image")
V2022_MASK = Path("/kaggle/input/datasets/rafiadabhi/beach-litter/mask")
BEPLI_DIR = Path("/kaggle/input/datasets/rafiadabhi/databepliv2")
BEPLI_JSON = BEPLI_DIR / "all_plastic_coco.json"
BEPLI_LIST = BEPLI_DIR / "pembagian_anotasi_bepli.md"
OUT = Path("/kaggle/working")

CKPT_INPUT = None
SKIP_TRAINING = CKPT_INPUT is not None

CLASSES = ["background", "artificial", "natural"]
PALETTE = np.array([[0, 0, 0], [255, 0, 0], [0, 255, 0]], dtype=np.int32)
NATURAL = 2
ID2LABEL = dict(enumerate(CLASSES))
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

PRETRAINED = "facebook/mask2former-swin-small-ade-semantic"
SEED = 42
CROP = 512
BATCH_SIZE = 4
ACCUM_STEPS = 2
WEIGHT_DECAY = 0.05
BACKBONE_LR_MULT = 0.1
POLY_POWER = 0.9
LR_CANDIDATES = [5e-5, 6e-5, 1e-4]
TUNE_EPOCHS = 8
FINAL_EPOCHS = 30
TIME_BUDGET_HOURS = 10.5
USE_AMP = DEVICE == "cuda"
NUM_WORKERS = 2
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

START_TIME = time.time()


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything()

## 3. v2022 data and split
Only files with a six-digit name (v2022) are used. The 70/10/20 split is stratified by quartile of natural-litter area and saved to `v2022_split.csv` for reuse in the main experiments.

In [ ]:
def rgb_to_index(mask_rgb: np.ndarray) -> np.ndarray:
    dist = ((mask_rgb[:, :, None, :].astype(np.int32) - PALETTE[None, None]) ** 2).sum(-1)
    return dist.argmin(-1).astype(np.uint8)


def read_rgb(path: Path) -> np.ndarray:
    return cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)


def read_label(path: Path, size: tuple[int, int] | None = None) -> np.ndarray:
    mask = read_rgb(path)
    if size is not None and mask.shape[:2] != size:
        mask = cv2.resize(mask, (size[1], size[0]), interpolation=cv2.INTER_NEAREST)
    return rgb_to_index(mask)


images = sorted(p for p in V2022_IMG.glob("*.jpg") if re.fullmatch(r"\d{6}", p.stem))
mask_by_stem = {p.stem: p for p in V2022_MASK.glob("*.png")}
pairs = [(p, mask_by_stem[p.stem]) for p in images if p.stem in mask_by_stem]
print(f"v2022 images: {len(images)} | with mask: {len(pairs)}")

rows = []
for img_path, mask_path in tqdm(pairs, desc="Class area"):
    small = cv2.resize(read_rgb(mask_path), (128, 128), interpolation=cv2.INTER_NEAREST)
    counts = np.bincount(rgb_to_index(small).ravel(), minlength=3) / small.shape[0] / small.shape[1]
    rows.append({"image": img_path.name, "mask": mask_path.name,
                 "artificial_pct": counts[1] * 100, "natural_pct": counts[2] * 100})
meta = pd.DataFrame(rows)
meta["strata"] = pd.qcut(meta["natural_pct"].rank(method="first"), 4, labels=False)

train_df, rest = train_test_split(meta, test_size=0.30, stratify=meta["strata"], random_state=SEED)
val_df, test_df = train_test_split(rest, test_size=2 / 3, stratify=rest["strata"], random_state=SEED)
split = pd.concat([train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")])
split.drop(columns="strata").sort_values("image").to_csv(OUT / "v2022_split.csv", index=False)

print(split.groupby("split")[["natural_pct", "artificial_pct"]].agg(["count", "mean"]).round(2))

## 4. Dataset and augmentation

In [ ]:
def train_transform() -> A.Compose:
    return A.Compose([
        A.SmallestMaxSize(max_size=CROP),
        A.RandomScale(scale_limit=(-0.5, 1.0), p=1.0),
        A.PadIfNeeded(min_height=CROP, min_width=CROP, border_mode=cv2.BORDER_REFLECT_101),
        A.RandomCrop(height=CROP, width=CROP),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.5),
        A.Normalize(mean=MEAN, std=STD),
    ])


def eval_transform() -> A.Compose:
    return A.Compose([A.SmallestMaxSize(max_size=CROP), A.Normalize(mean=MEAN, std=STD)])


class V2022Dataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform: A.Compose):
        self.items = [(V2022_IMG / r.image, V2022_MASK / r.mask) for r in df.itertuples()]
        self.transform = transform

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, i):
        img_path, mask_path = self.items[i]
        img = read_rgb(img_path)
        label = read_label(mask_path, img.shape[:2])
        out = self.transform(image=img, mask=label)
        image = torch.from_numpy(out["image"]).permute(2, 0, 1).float()
        return image, torch.from_numpy(out["mask"]).long()


def collate_train(batch):
    images = torch.stack([b[0] for b in batch])
    mask_labels, class_labels = [], []
    for _, label in batch:
        classes = torch.unique(label)
        mask_labels.append(torch.stack([(label == c).float() for c in classes]))
        class_labels.append(classes)
    return images, mask_labels, class_labels

## 5. Model, semantic prediction, and evaluation
Per-class probabilities are the query class scores weighted by the query masks. Evaluation reports per-class IoU at short side 512.

In [ ]:
def build_model() -> torch.nn.Module:
    return Mask2FormerForUniversalSegmentation.from_pretrained(
        PRETRAINED, id2label=ID2LABEL, label2id=LABEL2ID, ignore_mismatched_sizes=True
    ).to(DEVICE)


def semantic_probs(model, pixel_values: torch.Tensor) -> torch.Tensor:
    h, w = pixel_values.shape[-2:]
    pad_h, pad_w = (-h) % 32, (-w) % 32
    x = F.pad(pixel_values, (0, pad_w, 0, pad_h), mode="reflect") if pad_h or pad_w else pixel_values
    with torch.autocast(device_type="cuda", enabled=USE_AMP):
        out = model(pixel_values=x)
    cls = out.class_queries_logits.float().softmax(-1)[..., :-1]
    masks = F.interpolate(out.masks_queries_logits.float(), size=x.shape[-2:],
                          mode="bilinear", align_corners=False).sigmoid()
    return torch.einsum("bqc,bqhw->bchw", cls, masks)[..., :h, :w]


@torch.no_grad()
def evaluate(model, df: pd.DataFrame) -> np.ndarray:
    model.eval()
    loader = DataLoader(V2022Dataset(df, eval_transform()), batch_size=1, num_workers=NUM_WORKERS)
    conf = torch.zeros(3, 3, dtype=torch.long, device=DEVICE)
    for image, label in loader:
        image, label = image.to(DEVICE), label.to(DEVICE)
        pred = semantic_probs(model, image).argmax(1)
        conf += torch.bincount(label.flatten() * 3 + pred.flatten(), minlength=9).reshape(3, 3)
    conf = conf.double()
    tp = conf.diag()
    return (tp / (conf.sum(0) + conf.sum(1) - tp).clamp(min=1)).cpu().numpy()


def elapsed_hours() -> float:
    return (time.time() - START_TIME) / 3600

## 6. Training

In [ ]:
def train(lr: float, epochs: int, ckpt_path: Path | None = None) -> dict:
    seed_everything()
    model = build_model()
    loader = DataLoader(V2022Dataset(train_df, train_transform()), batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS, collate_fn=collate_train, drop_last=True, pin_memory=True)

    backbone = [p for n, p in model.named_parameters() if "pixel_level_module.encoder" in n]
    others = [p for n, p in model.named_parameters() if "pixel_level_module.encoder" not in n]
    optimizer = torch.optim.AdamW([
        {"params": backbone, "lr": lr * BACKBONE_LR_MULT},
        {"params": others, "lr": lr},
    ], weight_decay=WEIGHT_DECAY)

    total_steps = epochs * math.ceil(len(loader) / ACCUM_STEPS)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda step: (1 - min(step, total_steps) / total_steps) ** POLY_POWER)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    best = {"lr": lr, "natural_iou": -1.0, "epoch": 0, "val_iou": None}
    history = []
    for epoch in range(1, epochs + 1):
        if elapsed_hours() > TIME_BUDGET_HOURS:
            print(f"Time budget reached, stopped after epoch {epoch - 1}.")
            break
        model.train()
        running, n_steps = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        for step, (images, mask_labels, class_labels) in enumerate(tqdm(loader, desc=f"lr={lr:g} ep{epoch}", leave=False)):
            images = images.to(DEVICE, non_blocking=True)
            mask_labels = [m.to(DEVICE) for m in mask_labels]
            class_labels = [c.to(DEVICE) for c in class_labels]
            with torch.autocast(device_type="cuda", enabled=USE_AMP):
                loss = model(pixel_values=images, mask_labels=mask_labels, class_labels=class_labels).loss
            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue
            scaler.scale(loss / ACCUM_STEPS).backward()
            if (step + 1) % ACCUM_STEPS == 0 or step + 1 == len(loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
            running += loss.item()
            n_steps += 1

        val_iou = evaluate(model, val_df)
        history.append({"epoch": epoch, "loss": running / max(n_steps, 1),
                        **{f"val_iou_{c}": v for c, v in zip(CLASSES, val_iou)}})
        print(f"lr={lr:g} | epoch {epoch}/{epochs} | loss {running / max(n_steps, 1):.4f} | "
              f"val IoU bg/art/nat = {val_iou.round(4)} | {elapsed_hours():.2f} h")
        if val_iou[NATURAL] > best["natural_iou"]:
            best.update(natural_iou=float(val_iou[NATURAL]), epoch=epoch, val_iou=val_iou.tolist())
            if ckpt_path is not None:
                torch.save(model.state_dict(), ckpt_path)

    best["history"] = history
    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()
    return best

## 7. Learning-rate tuning
Each candidate is trained for `TUNE_EPOCHS` epochs; the one with the best validation IoU on the natural class is kept.

In [ ]:
if SKIP_TRAINING:
    BEST_LR = None
    print("Tuning skipped, using checkpoint:", CKPT_INPUT)
else:
    tuning = []
    for lr in LR_CANDIDATES:
        result = train(lr, TUNE_EPOCHS)
        tuning.append(result)
        print(f">> lr={lr:g}: best natural IoU {result['natural_iou']:.4f} (epoch {result['epoch']})\n")

    tuning_df = pd.DataFrame([{k: v for k, v in r.items() if k != "history"} for r in tuning])
    tuning_df.to_csv(OUT / "tuning_lr.csv", index=False)
    BEST_LR = float(tuning_df.sort_values("natural_iou", ascending=False).iloc[0]["lr"])
    print(tuning_df[["lr", "natural_iou", "epoch"]])
    print("Selected learning rate:", BEST_LR)

## 8. Final training and test evaluation
The checkpoint with the best validation natural IoU is kept and evaluated once on the test set.

In [ ]:
if SKIP_TRAINING:
    CKPT = Path(CKPT_INPUT)
    final = {"epoch": None, "val_iou": None}
else:
    CKPT = OUT / "mask2former_v2022_best.pt"
    final = train(BEST_LR, FINAL_EPOCHS, ckpt_path=CKPT)
    assert CKPT.exists(), "No checkpoint saved, probably out of time during tuning. Reduce TUNE_EPOCHS or FINAL_EPOCHS."
    pd.DataFrame(final["history"]).to_csv(OUT / "history_final.csv", index=False)
    print(f"Best checkpoint: epoch {final['epoch']} | val natural IoU {final['natural_iou']:.4f}")

model = build_model()
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
val_iou = evaluate(model, val_df)
test_iou = evaluate(model, test_df)

metrics = {
    "pretrained": PRETRAINED, "checkpoint": str(CKPT), "lr": BEST_LR, "best_epoch": final["epoch"],
    "val_iou": dict(zip(CLASSES, val_iou.tolist())),
    "test_iou": dict(zip(CLASSES, test_iou.tolist())),
    "test_miou": float(test_iou.mean()),
}
(OUT / "metrics_mask2former.json").write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

## 9. BePLi pseudo-labels
Natural comes from the Mask2Former prediction; artificial comes only from BePLi's plastic annotations (the model's artificial prediction is ignored). Plastic always overrides natural. Masks are saved as single-channel PNG (0/1/2) at the original image size.

In [ ]:
def load_coco_index(json_path: Path) -> dict:
    data = json.loads(json_path.read_text(encoding="utf-8"))
    id_to_name = {img["id"]: Path(img["file_name"]).name for img in data["images"]}
    index = {name: {"h": img["height"], "w": img["width"], "segs": []}
             for img in data["images"] for name in [Path(img["file_name"]).name]}
    for ann in data["annotations"]:
        index[id_to_name[ann["image_id"]]]["segs"].append(ann["segmentation"])
    return index


def decode_segmentation(seg, h: int, w: int) -> np.ndarray:
    if isinstance(seg, list):
        rle = mask_utils.merge(mask_utils.frPyObjects(seg, h, w))
    elif isinstance(seg.get("counts"), list):
        rle = mask_utils.frPyObjects(seg, h, w)
    else:
        rle = seg
    m = mask_utils.decode(rle)
    return m.max(axis=2) if m.ndim == 3 else m


def plastic_mask(info: dict | None, shape: tuple[int, int]) -> np.ndarray:
    mask = np.zeros(shape, dtype=np.uint8)
    if info is None:
        return mask
    for seg in info["segs"]:
        m = decode_segmentation(seg, info["h"], info["w"])
        if m.shape != shape:
            m = cv2.resize(m, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST)
        mask |= m.astype(np.uint8)
    return mask


@torch.no_grad()
def predict_natural(model, img: np.ndarray) -> np.ndarray:
    h, w = img.shape[:2]
    x = eval_transform()(image=img)["image"]
    x = torch.from_numpy(x).permute(2, 0, 1)[None].float().to(DEVICE)
    probs = semantic_probs(model, x)
    probs = F.interpolate(probs, size=(h, w), mode="bilinear", align_corners=False)
    return (probs.argmax(1)[0] == NATURAL).cpu().numpy()


def read_bepli_list(md_path: Path) -> list[str]:
    blocks = re.findall(r"```(.*?)```", md_path.read_text(encoding="utf-8"), re.S)
    numbers = {n for block in blocks for n in re.findall(r"\b\d{6}\b", block)}
    return [f"{n}.png" for n in sorted(numbers)]


def index_images(root: Path, preferred: str = "original_images") -> dict:
    index = {}
    for p in sorted(root.rglob("*.png")):
        if p.name not in index or preferred in p.parts:
            index[p.name] = p
    return index


bepli_files = read_bepli_list(BEPLI_LIST)
coco = load_coco_index(BEPLI_JSON)
bepli_paths = index_images(BEPLI_DIR)
missing = [f for f in bepli_files if f not in bepli_paths]
print(f"BePLi images: {len(bepli_files)} | with plastic annotation: {sum(f in coco for f in bepli_files)}"
      f" | image files found: {len(bepli_files) - len(missing)}")
if missing:
    example = next(iter(bepli_paths.values()), None)
    raise FileNotFoundError(f"{len(missing)} images not found, e.g. {missing[:5]}. "
                            f"Example PNG in the dataset: {example}")
print("Image folder:", bepli_paths[bepli_files[0]].parent)

PSEUDO_DIR = OUT / "bepli_pseudo_masks"
PSEUDO_DIR.mkdir(exist_ok=True)
model.eval()

stats = []
for name in tqdm(bepli_files, desc="Pseudo-label"):
    img = read_rgb(bepli_paths[name])
    natural = predict_natural(model, img)
    plastic = plastic_mask(coco.get(name), img.shape[:2])
    label = np.zeros(img.shape[:2], dtype=np.uint8)
    label[natural] = 2
    label[plastic == 1] = 1
    cv2.imwrite(str(PSEUDO_DIR / name), label)
    stats.append({"file": name, "artificial_pct": (label == 1).mean() * 100,
                  "natural_pct": (label == 2).mean() * 100})

stats_df = pd.DataFrame(stats)
stats_df.to_csv(OUT / "bepli_pseudo_stats.csv", index=False)
print(stats_df.describe().round(3))
print(f"Images with natural > 0: {(stats_df.natural_pct > 0).sum()} of {len(stats_df)}")

## 10. QC overlays
Thirty random images are saved to `qc_overlay/` (left: original; right: red = artificial, green = natural).

In [ ]:
QC_DIR = OUT / "qc_overlay"
QC_DIR.mkdir(exist_ok=True)
COLORS = np.array([[0, 0, 0], [255, 0, 0], [0, 255, 0]], dtype=np.float32)

qc_files = random.Random(SEED).sample(bepli_files, min(30, len(bepli_files)))
for name in qc_files:
    img = read_rgb(bepli_paths[name])
    label = cv2.imread(str(PSEUDO_DIR / name), cv2.IMREAD_GRAYSCALE)
    overlay = img.astype(np.float32)
    fg = label > 0
    overlay[fg] = 0.45 * overlay[fg] + 0.55 * COLORS[label[fg]]
    scale = min(1.0, 1200 / img.shape[1])
    size = (round(img.shape[1] * scale), round(img.shape[0] * scale))
    pair = np.concatenate([cv2.resize(img, size), np.full((size[1], 10, 3), 255, np.uint8),
                           cv2.resize(overlay.astype(np.uint8), size)], axis=1)
    cv2.imwrite(str(QC_DIR / f"{Path(name).stem}_qc.jpg"), cv2.cvtColor(pair, cv2.COLOR_RGB2BGR))

fig, axes = plt.subplots(6, 1, figsize=(14, 26))
for ax, name in zip(axes, qc_files[:6]):
    ax.imshow(cv2.cvtColor(cv2.imread(str(QC_DIR / f"{Path(name).stem}_qc.jpg")), cv2.COLOR_BGR2RGB))
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 11. Archive
`bepli_pseudo_masks.zip` and `qc_overlay.zip` are written to the output folder.

In [ ]:
shutil.make_archive(str(OUT / "bepli_pseudo_masks"), "zip", PSEUDO_DIR)
shutil.make_archive(str(OUT / "qc_overlay"), "zip", QC_DIR)
print(f"Finished in {elapsed_hours():.2f} h")
for f in sorted(OUT.glob("*")):
    if f.is_file():
        print(f"{f.name:40s} {f.stat().st_size / 1e6:8.1f} MB")